<a href="https://colab.research.google.com/github/sushmitamalakar10/LLM/blob/main/PDF_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf

In [ ]:
!pip install transformers

In [ ]:
from pypdf import PdfReader
from transformers import pipeline

In [ ]:
# !pip uninstall -y transformers
!pip install transformers==4.37.2


## Loading Model

In [ ]:
from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
  login(token=HF_TOKEN)
  print("Logged in")


else:
  print("Token not set")

Logged in


In [ ]:
# !pip uninstall -y transformers tensorflow huggingface-hub


Found existing installation: transformers 4.37.2
Uninstalling transformers-4.37.2:
  Successfully uninstalled transformers-4.37.2
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2


In [ ]:
# # Uninstall conflicting packages
# !pip uninstall -y transformers huggingface-hub tensorflow

# # Install compatible versions
# !pip install transformers==4.37.2 torch huggingface-hub==0.36.2


  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.37.2-py3-none-any.whl (8.4 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.


In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# # to show the pdf content

# def show_pdf_content(pdf_file):
#   if pdf_file is None:
#     return "Please upload PDF file."

#   reader = PdfReader(pdf_file)
#   text = ""

#   for page in reader.pages:
#     page_text = page.extract_text()

#     if page_text:
#       text += page_text + "\n"

#   return text if text else "No readable text found in this PDF file"

In [ ]:
# Extract Text from PDF
def extract_text_from_pdf(pdf_file):
  reader = PdfReader(pdf_file)

  text = ""

  for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
      text += page_text + "\n"

  return text

In [ ]:
pdf_text = extract_text_from_pdf("/content/SushmitaMalakar_CV.pdf")
pdf_text

' \nSUSHMITA MALAKAR  \nDATA SCIENCE  & MACHINE LEARNING  ENTHUSIAST  \n9818085057 | sushmalakar10@gmail.com | Satungal, Kathmandu \nwww.linkedin.com/in/sushmita-malakar-a3a5a9247 \nwww.github.com/sushmitamalakar10   \n \n \nABOUT ME \nData Science and Machine Learning enthusiast with hands -on experience in Python, data analysis, and \npredictive modeling. Skilled in data preprocessing, exploratory data analysis, visualization, and machine \nlearning algorithms. Experienced in building end -to-end ML projects and deploying models using \nStreamlit. Seeking opportunities to apply analytical and technical skills to solve real -world problems in \ncollaborative environments. \n \nSKILLS \nProgramming & Tools Python, SQL, Pandas, NumPy, Matplotlib, Seaborn, Scikit -learn, TensorFlow, \nKeras, XGBoost, Flask, Streamlit, GitHub, Jupyter Notebook, Google Colab \nMachine Learning Supervised and Unsupervised Learning,  Feature Engineering, Model \nEvaluation, Hyperparameter Tuning, Natural Lan

In [ ]:
def chunk_text(text, chunk_size=800):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

In [ ]:
chunks = chunk_text(pdf_text, chunk_size=800)             # split into 800-word chunks


In [ ]:
chunks

['SUSHMITA MALAKAR DATA SCIENCE & MACHINE LEARNING ENTHUSIAST 9818085057 | sushmalakar10@gmail.com | Satungal, Kathmandu www.linkedin.com/in/sushmita-malakar-a3a5a9247 www.github.com/sushmitamalakar10 ABOUT ME Data Science and Machine Learning enthusiast with hands -on experience in Python, data analysis, and predictive modeling. Skilled in data preprocessing, exploratory data analysis, visualization, and machine learning algorithms. Experienced in building end -to-end ML projects and deploying models using Streamlit. Seeking opportunities to apply analytical and technical skills to solve real -world problems in collaborative environments. SKILLS Programming & Tools Python, SQL, Pandas, NumPy, Matplotlib, Seaborn, Scikit -learn, TensorFlow, Keras, XGBoost, Flask, Streamlit, GitHub, Jupyter Notebook, Google Colab Machine Learning Supervised and Unsupervised Learning, Feature Engineering, Model Evaluation, Hyperparameter Tuning, Natural Language Processing (NLP) , Neural Networks, Deep L

In [ ]:
import gradio as gr

In [ ]:
def summarize_pdf(pdf_file):
    if pdf_file is None:
        return "Please upload a PDF file."

    text = extract_text_from_pdf(pdf_file)
    chunks = chunk_text(text)

    summaries = []
    for chunk in chunks:
        summary = summarizer(
            chunk,
            max_length=150,
            min_length=40,
            do_sample=False
        )
        summaries.append(summary[0]["summary_text"])

    final_summary = " ".join(summaries)
    return final_summary


In [ ]:
# summarize_pdf("/content/SushmitaMalakar_CV.pdf")

'Data Science and Machine Learning enthusiast with hands -on experience in Python, data analysis, and predictive modeling. Experienced in building end -to-end ML projects and deploying models using Streamlit. Seeking opportunities to apply analytical and technical skills to solve real -world problems in collaborative environments.'

In [ ]:
interface = gr.Interface(
    fn=summarize_pdf,
    inputs=gr.File(file_types=[".pdf"]),
    outputs=gr.Textbox(lines=20, placeholder="Summary will appear here"),
    title="📄 PDF Summarizer",
    description="Upload a PDF and get a summary."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://960167074fadd5b447.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
|# # Save locally
# summarizer.model.save_pretrained("./my_pdf_summarizer_model")
# summarizer.tokenizer.save_pretrained("./my_pdf_summarizer_model")

# Reload later
# from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
# model = AutoModelForSeq2SeqLM.from_pretrained("./my_pdf_summarizer_model")
# tokenizer = AutoTokenizer.from_pretrained("./my_pdf_summarizer_model")
# summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)


In [ ]:
!pip list

Package                                  Version
---------------------------------------- -----------------
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.3
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                           

In [ ]:
!pip freeze > requirements.txt

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import login
import os

token = os.getenv("HF_TOKEN")
login(token=token)
